# Modelos com validação temporal, tuning por ROC-AUC e walk-forward

Este notebook implementa protocolo mais rigoroso para comparação entre modelos de classificação em série temporal.

Fluxo:

1. **Ordenação temporal global** do dataset
2. **Holdout final cego**: último bloco temporal fica reservado só para avaliação final
3. **Tuning de hiperparâmetros por ROC-AUC** usando dois protocolos no período pré-teste:
   - `TimeSeriesSplit` expansivo
   - `Walk-forward` com janela fixa
4. **Purge gap de 15 minutos** entre treino e validação/teste para evitar leakage pelo horizonte do target
5. **Avaliação final** no holdout com métricas de ranking e classificação

Objetivo: prever se o preço de um contrato Polymarket **sobe nos próximos 15 minutos** (`target = 1`).


In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
from time import perf_counter

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


## Carregamento e preparação

Preferência por `features/ml_features_v3.parquet`. Se ele não existir, o notebook gera `lags` e `ofi_corrected` a partir de `ml_features_1m_v2.parquet`.


In [ ]:
TARGET_HORIZON_MINUTES = 15
PURGE_GAP = timedelta(minutes=TARGET_HORIZON_MINUTES)
TEST_START = datetime(2026, 3, 10, tzinfo=timezone.utc)
TEST_END = datetime(2026, 3, 12, tzinfo=timezone.utc)
WALK_FORWARD_TRAIN_DAYS = 2
WALK_FORWARD_VALID_DAYS = 1
N_TSCV_SPLITS = 3
RANDOM_STATE = 42

FEATURES = [
    "close_mid",
    "depth_imbalance",
    "mean_spread",
    "close_spread",
    "bar_volatility",
    "ofi_corrected",
    "close_mid_lag1",
    "depth_imbalance_lag1",
    "mean_spread_lag1",
    "close_spread_lag1",
    "bar_volatility_lag1",
    "ofi_corrected_lag1",
]

LAG_COLS = [
    "close_mid_lag1",
    "depth_imbalance_lag1",
    "mean_spread_lag1",
    "close_spread_lag1",
    "bar_volatility_lag1",
    "ofi_corrected_lag1",
]


def build_v3_from_v2(raw: pl.DataFrame) -> pl.DataFrame:
    df = raw.sort(["market_id", "minute_bar"])
    df = df.with_columns(
        pl.when(pl.col("total_volume") > 0)
        .then((pl.col("buy_volume") - pl.col("sell_volume")) / pl.col("total_volume"))
        .otherwise(0.0)
        .alias("ofi_corrected")
    )
    base_cols = [
        "close_mid",
        "depth_imbalance",
        "mean_spread",
        "close_spread",
        "bar_volatility",
        "ofi_corrected",
    ]
    return df.with_columns(
        [pl.col(col).shift(1).over("market_id").alias(f"{col}_lag1") for col in base_cols]
    )


def load_dataset() -> pl.DataFrame:
    candidates = [
        Path("features/ml_features_v3.parquet"),
        Path("features/ml_features_1m_v2.parquet"),
        Path("ml_features_1m_v2.parquet"),
    ]
    for path in candidates:
        if not path.exists():
            continue
        raw = pl.read_parquet(path)
        if "close_mid_lag1" in raw.columns:
            print(f"Carregado: {path} (já com lags)")
            return raw
        print(f"Carregado: {path} — gerando lags e ofi_corrected...")
        return build_v3_from_v2(raw)
    raise FileNotFoundError("Nenhum parquet de features encontrado.")


df = load_dataset()
df = df.sort(["minute_bar", "market_id"]).drop_nulls(LAG_COLS)

print(df.shape)
print(df.select(pl.min("minute_bar").alias("inicio"), pl.max("minute_bar").alias("fim")))


## Protocolo de divisão temporal

- `test final`: `10/03/2026` até `11/03/2026 23:59 UTC`
- `train + tuning`: tudo antes de `10/03/2026`
- `purge gap = 15 min`: remove linhas no fim do treino cujo rótulo usa preços já dentro do próximo bloco

Isso evita dois problemas:

- `TimeSeriesSplit` em dados fora de ordem temporal
- vazamento pelo próprio target, já que o alvo olha **15 minutos à frente**


In [ ]:
pre_test = df.filter(pl.col("minute_bar") < TEST_START)
final_test = df.filter((pl.col("minute_bar") >= TEST_START) & (pl.col("minute_bar") < TEST_END))
pre_test = pre_test.filter(pl.col("minute_bar") < TEST_START - PURGE_GAP)

print("Pré-teste:", pre_test.shape)
print("Teste final:", final_test.shape)
print(pre_test.select(pl.min("minute_bar").alias("inicio"), pl.max("minute_bar").alias("fim")))
print(final_test.select(pl.min("minute_bar").alias("inicio"), pl.max("minute_bar").alias("fim")))

baseline_acc = float((final_test["target"] == 0).mean())
print(f"Baseline de accuracy no teste final (sempre classe 0): {baseline_acc:.4f}")


In [ ]:
def to_xy(frame: pl.DataFrame):
    X = frame.select(FEATURES).to_pandas().reset_index(drop=True)
    y = frame["target"].to_numpy().ravel()
    bars = frame["minute_bar"].to_numpy()
    return X, y, bars


X_pre, y_pre, bars_pre = to_xy(pre_test)
X_test, y_test, bars_test = to_xy(final_test)


## Modelos e grade de hiperparâmetros

`ROC-AUC` será métrica de tuning. Ela usa `y_proba`, não `y_pred`.


In [ ]:
PARAM_GRIDS = {
    "LightGBM": [
        {"n_estimators": [100], "max_depth": [4], "learning_rate": [0.05]},
        {"n_estimators": [200], "max_depth": [4], "learning_rate": [0.05]},
        {"n_estimators": [100], "max_depth": [6], "learning_rate": [0.05]},
        {"n_estimators": [200], "max_depth": [6], "learning_rate": [0.05]},
    ],
    "Decision Tree": [
        {"max_depth": [4], "min_samples_leaf": [1]},
        {"max_depth": [6], "min_samples_leaf": [1]},
        {"max_depth": [6], "min_samples_leaf": [5]},
        {"max_depth": [10], "min_samples_leaf": [10]},
    ],
    "Logistic Regression": [
        {"C": [0.01]},
        {"C": [0.1]},
        {"C": [1.0]},
        {"C": [10.0]},
    ],
    "Random Forest": [
        {"n_estimators": [50], "max_depth": [6], "min_samples_leaf": [1]},
        {"n_estimators": [100], "max_depth": [6], "min_samples_leaf": [1]},
        {"n_estimators": [50], "max_depth": [10], "min_samples_leaf": [5]},
        {"n_estimators": [100], "max_depth": [10], "min_samples_leaf": [5]},
    ],
}


def make_model(name: str, **params):
    if name == "LightGBM":
        return lgb.LGBMClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
            verbose=-1,
            **params,
        )
    if name == "Decision Tree":
        return DecisionTreeClassifier(
            random_state=RANDOM_STATE,
            class_weight="balanced",
            **params,
        )
    if name == "Logistic Regression":
        return LogisticRegression(
            solver="lbfgs",
            class_weight="balanced",
            max_iter=1000,
            random_state=RANDOM_STATE,
            **params,
        )
    if name == "Random Forest":
        return RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
            **params,
        )
    raise ValueError(name)


## Folds temporais

- `TimeSeriesSplit`: janela expansiva no período pré-teste
- `Walk-forward`: `2 dias` de treino fixo + `1 dia` de validação, avançando no tempo
- ambos aplicam `purge gap = 15 min`


In [ ]:
def build_walk_forward_splits(bars, train_days=2, valid_days=1, gap=PURGE_GAP):
    df_time = pd.DataFrame({"minute_bar": pd.to_datetime(bars, utc=True)})
    all_days = sorted(df_time["minute_bar"].dt.normalize().unique())
    splits = []
    for start_idx in range(len(all_days) - train_days):
        train_start = all_days[start_idx]
        train_end_exclusive = train_start + pd.Timedelta(days=train_days)
        valid_start = train_end_exclusive
        valid_end_exclusive = valid_start + pd.Timedelta(days=valid_days)

        train_mask = (
            (df_time["minute_bar"] >= train_start)
            & (df_time["minute_bar"] < valid_start - pd.Timedelta(gap))
        )
        valid_mask = (
            (df_time["minute_bar"] >= valid_start)
            & (df_time["minute_bar"] < valid_end_exclusive)
        )

        train_idx = np.flatnonzero(train_mask.to_numpy())
        valid_idx = np.flatnonzero(valid_mask.to_numpy())
        if len(train_idx) and len(valid_idx):
            splits.append((train_idx, valid_idx, str(train_start.date()), str(valid_start.date())))
    return splits


def build_tscv_splits(X, bars, n_splits=3, gap=TARGET_HORIZON_MINUTES):
    splitter = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    splits = []
    for fold, (train_idx, valid_idx) in enumerate(splitter.split(X), start=1):
        train_end = pd.to_datetime(bars[train_idx[-1]], utc=True)
        valid_start = pd.to_datetime(bars[valid_idx[0]], utc=True)
        splits.append((train_idx, valid_idx, f"fold_{fold}", f"{train_end} -> {valid_start}"))
    return splits


walk_splits = build_walk_forward_splits(bars_pre)
tscv_splits = build_tscv_splits(X_pre, bars_pre, n_splits=N_TSCV_SPLITS)

print(f"Walk-forward folds: {len(walk_splits)}")
print(f"TimeSeriesSplit folds: {len(tscv_splits)}")
if walk_splits:
    print("Exemplo walk-forward:", walk_splits[0][2:])
if tscv_splits:
    print("Exemplo TSCV:", tscv_splits[0][2:])


In [ ]:
def score_split(model_name, params, X_train, y_train, X_valid, y_valid):
    model = make_model(model_name, **params)
    if model_name == "Logistic Regression":
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_valid_scaled = scaler.transform(X_valid)
        model.fit(X_train_scaled, y_train)
        y_proba = model.predict_proba(X_valid_scaled)[:, 1]
        return roc_auc_score(y_valid, y_proba)
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_valid)[:, 1]
    return roc_auc_score(y_valid, y_proba)


def evaluate_grid(model_name, X, y, splits):
    rows = []
    for params in ParameterGrid(PARAM_GRIDS[model_name]):
        fold_aucs = []
        for train_idx, valid_idx, _, _ in splits:
            auc_value = score_split(
                model_name,
                params,
                X.iloc[train_idx],
                y[train_idx],
                X.iloc[valid_idx],
                y[valid_idx],
            )
            fold_aucs.append(auc_value)
        rows.append(
            {
                "model": model_name,
                "params": params,
                "mean_auc": float(np.mean(fold_aucs)),
                "std_auc": float(np.std(fold_aucs)),
                "fold_aucs": fold_aucs,
            }
        )
    return sorted(rows, key=lambda row: row["mean_auc"], reverse=True)


def evaluate_protocol(splits, protocol_name):
    best = {}
    ranking = []
    started_at = perf_counter()
    for model_name in PARAM_GRIDS:
        results = evaluate_grid(model_name, X_pre, y_pre, splits)
        best[model_name] = results[0]
        ranking.append(
            {
                "Modelo": model_name,
                "Combinações": len(results),
                "Folds": len(splits),
                "Fits totais": len(results) * len(splits),
                "Melhor ROC-AUC": results[0]["mean_auc"],
                "Std": results[0]["std_auc"],
                "Melhores hiperparâmetros": str(results[0]["params"]),
            }
        )
    elapsed = perf_counter() - started_at
    summary = pd.DataFrame(ranking).sort_values("Melhor ROC-AUC", ascending=False)
    print(f"{protocol_name} concluído em {elapsed:.1f}s")
    print(f"Combinações totais avaliadas: {summary['Combinações'].sum()}")
    print(f"Fits totais: {summary['Fits totais'].sum()}")
    return best, summary, elapsed


best_tscv, tscv_df, tscv_elapsed = evaluate_protocol(tscv_splits, "TimeSeriesSplit")
best_walk, walk_df, walk_elapsed = evaluate_protocol(walk_splits, "Walk-forward")

print("Melhores combinações — TimeSeriesSplit")
display(tscv_df)
print("Melhores combinações — Walk-forward")
display(walk_df)


## Comparação entre protocolos de validação

Aqui a comparação ainda acontece **sem tocar no teste final**. O objetivo é ver como cada protocolo ranqueia os hiperparâmetros e os modelos.


In [ ]:
comparison_rows = []
for model_name in PARAM_GRIDS:
    comparison_rows.append(
        {
            "Modelo": model_name,
            "ROC-AUC TSCV": best_tscv[model_name]["mean_auc"],
            "ROC-AUC Walk-forward": best_walk[model_name]["mean_auc"],
            "Melhor params TSCV": str(best_tscv[model_name]["params"]),
            "Melhor params Walk-forward": str(best_walk[model_name]["params"]),
        }
    )
comparison_df = pd.DataFrame(comparison_rows).sort_values("ROC-AUC Walk-forward", ascending=False)
search_space_df = pd.DataFrame(
    [
        {"Versão": "Antes", "Combinações totais": 22},
        {"Versão": "Depois", "Combinações totais": sum(len(list(ParameterGrid(grid))) for grid in PARAM_GRIDS.values())},
    ]
)
search_space_df["Redução vs antes"] = search_space_df["Combinações totais"].apply(
    lambda value: f"{(1 - value / 22):.0%}" if value <= 22 else "0%"
)
timing_df = pd.DataFrame(
    [
        {"Protocolo": "TimeSeriesSplit", "Tempo (s)": tscv_elapsed, "Combinações por modelo": 4},
        {"Protocolo": "Walk-forward", "Tempo (s)": walk_elapsed, "Combinações por modelo": 4},
    ]
)
display(search_space_df)
display(timing_df)
comparison_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = comparison_df.set_index("Modelo")[["ROC-AUC TSCV", "ROC-AUC Walk-forward"]]
plot_df.plot(kind="bar", ax=ax, color=["#1f77b4", "#ff7f0e"])
ax.set_title("ROC-AUC médio na validação temporal")
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0.5, 1.0)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Treino final com melhores hiperparâmetros do walk-forward

Como `walk-forward` é método temporal mais próximo de operação real aqui, usamos seus melhores hiperparâmetros para treinar no período pré-teste inteiro e avaliar no holdout final.


In [ ]:
def fit_and_evaluate(model_name, params):
    model = make_model(model_name, **params)
    scaler = None
    if model_name == "Logistic Regression":
        scaler = StandardScaler()
        X_pre_fit = scaler.fit_transform(X_pre)
        X_test_eval = scaler.transform(X_test)
        model.fit(X_pre_fit, y_pre)
        y_proba = model.predict_proba(X_test_eval)[:, 1]
    else:
        model.fit(X_pre, y_pre)
        y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    return {
        "model": model,
        "scaler": scaler,
        "metrics": {
            "Modelo": model_name,
            "ROC-AUC": roc_auc_score(y_test, y_proba),
            "PR-AUC": auc(recall, precision),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        },
        "y_pred": y_pred,
        "y_proba": y_proba,
    }


final_runs = {}
final_rows = []
for model_name, result in best_walk.items():
    run = fit_and_evaluate(model_name, result["params"])
    final_runs[model_name] = run
    final_rows.append(run["metrics"])

holdout_df = pd.DataFrame(final_rows).sort_values("ROC-AUC", ascending=False)
holdout_df


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

for name, run in final_runs.items():
    y_proba = run["y_proba"]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    ax1.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc_score(y_test, y_proba):.3f})")
    ax2.plot(recall, precision, lw=2, label=f"{name} (PR-AUC = {auc(recall, precision):.3f})")

ax1.plot([0, 1], [0, 1], "k--", lw=1, label="Aleatório (AUC = 0.5)")
ax1.set_title("ROC Curve — holdout final")
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.legend()

ax2.set_title("Precision-Recall Curve — holdout final")
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.legend()

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(final_runs), figsize=(15, 4))
if len(final_runs) == 1:
    axes = [axes]

for ax, (name, run) in zip(axes, final_runs.items()):
    cm = confusion_matrix(y_test, run["y_pred"])
    ConfusionMatrixDisplay(confusion_matrix=cm).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)

plt.suptitle("Matrizes de confusão — holdout final", y=1.02)
plt.tight_layout()
plt.show()


## Interpretação de `y_pred`, `y_proba`, ROC e AUC

- `y_proba`: probabilidade estimada de classe `1` para cada linha. Ex.: `0.82`.
- `y_pred`: classe final depois de aplicar threshold sobre `y_proba`. Com threshold `0.5`, `0.82 -> 1`, `0.31 -> 0`.
- Curva ROC: construída **variando threshold**, não hiperparâmetro.
- Cada threshold gera um par:
  - `TPR` = taxa de verdadeiros positivos (`recall`)
  - `FPR` = taxa de falsos positivos
- `AUC`: área sob curva ROC. Interpretação prática: quão bem modelo ranqueia positivos acima de negativos.
- `AUC = 0.5`: ranking quase aleatório.
- `AUC = 1.0`: separação perfeita.

Hiperparâmetros afetam modelo treinado e, por consequência, mudam `y_proba`. Mas curva ROC em si nasce da variação do threshold aplicado nesses scores.


## Síntese

- `TimeSeriesSplit` continua útil, mas agora roda sobre dados ordenados e com `gap`.
- `Walk-forward` foi adicionado como protocolo separado: treino fixo de `2 dias`, validação de `1 dia`.
- Hiperparâmetros são escolhidos por `ROC-AUC` só no período pré-teste.
- `Test final` fica intocado até fim do processo.
- `ROC-AUC` usa `y_proba` e mede capacidade de ranking; métricas thresholdadas usam `y_pred`.
- `baseline_acc` vale apenas para `accuracy`; ROC tem baseline natural em `0.5`.
